# GVH Diagonal Cubic — RACC-C.4-real.2.5
## GW150914 H1/L1 — Cross-Validated Detector Transfer Convergence Audit

**Auteur :** Charlemagne O Laurince  
**Étape :** `RACC-C.4-real.2.5`  
**Noyau GVH Diagonal modifié : NON**

### Objectif

Tester si le transfert relatif complexe H1→L1 identifié dans `real.2.4` explique réellement la divergence inter-estimateurs **hors-échantillon**.

Principe :

\[
\boxed{
T(f)\ \text{appris sur train}
\rightarrow
\text{appliqué sur test}
}
\]

puis inversion :

\[
\boxed{
T(f)\ \text{appris sur test}
\rightarrow
\text{appliqué sur train}
}
\]

Aucune valeur cible du lag n'est utilisée.

### Question falsifiable

La réduction du spread

\[
\Delta\tau_{\rm estimators}
=
\max(\tau_i)-\min(\tau_i)
\]

survit-elle lorsque \(T(f)\) est appris sur une portion indépendante du signal ?

### Stop-rule

Même en cas de succès :

\[
\texttt{TIMING-VALIDATED=False},
\qquad
D_T^{ref}=\texttt{NOT-AUTHORIZED},
\qquad
\texttt{REAL3-AUTHORIZED=False}
\]

dans ce notebook.

Un test final séparé sera requis avant toute autorisation de `real.3`.

**Notebook précédent de référence :** MD5 `8a746d753fd7084bce0c2377f01e4d21`.


In [1]:

import os,sys,subprocess,importlib,json
from pathlib import Path

def pip_install(pkg):
    subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])

try:
    import lalframe
except Exception:
    pip_install("lalsuite")
    importlib.invalidate_caches()
    import lalframe

try:
    import gwpy
except Exception:
    pip_install("gwpy")
    importlib.invalidate_caches()
    import gwpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from gwpy.timeseries import TimeSeries

print("gwpy:",gwpy.__version__)


/usr/local/lib/python3.12/dist-packages/lalframe/_lalframe_swig.py:8: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


gwpy: 4.0.1


## 1 — Chargement des données réelles

In [2]:

GPS_EVENT=1126259462.0
FS_EXPECTED=4096.0

def discover(pattern):
    roots=[Path.cwd(),Path("/content"),Path("/content/drive/MyDrive"),Path("/mnt/data")]
    hits=[]
    for root in roots:
        if not root.exists():
            continue
        for d in [root,root/"data"/"GW150914",root/"gvh_diagonal_cubic"/"data"/"GW150914"]:
            if d.exists():
                hits += list(d.glob(pattern))
    return hits[0].resolve() if hits else None

H1=discover("H-H1_LOSC_4_V2-1126259446-32*.gwf")
L1=discover("L-L1_LOSC_4_V2-1126259446-32*.gwf")

if H1 is None or L1 is None:
    try:
        from google.colab import files
        files.upload()
        H1=discover("H-H1_LOSC_4_V2-1126259446-32*.gwf")
        L1=discover("L-L1_LOSC_4_V2-1126259446-32*.gwf")
    except Exception:
        pass

if H1 is None or L1 is None:
    raise FileNotFoundError("H1/L1 introuvables.")

h1_ts=TimeSeries.read(str(H1),channel="H1:LOSC-STRAIN")
l1_ts=TimeSeries.read(str(L1),channel="L1:LOSC-STRAIN")

h1=np.asarray(h1_ts.value,float)
l1=np.asarray(l1_ts.value,float)
fs=float(h1_ts.sample_rate.value)
gps0=float(h1_ts.t0.value)
t_rel=gps0+np.arange(len(h1))/fs-GPS_EVENT

assert np.isclose(fs,FS_EXPECTED)
assert len(h1)==len(l1)
assert np.isfinite(h1).all() and np.isfinite(l1).all()

print(H1.name)
print(L1.name)
print("N =",len(h1),"fs =",fs)


Saving H-H1_LOSC_4_V2-1126259446-32 2.gwf to H-H1_LOSC_4_V2-1126259446-32 2.gwf
Saving L-L1_LOSC_4_V2-1126259446-32.gwf to L-L1_LOSC_4_V2-1126259446-32.gwf
H-H1_LOSC_4_V2-1126259446-32 2.gwf
L-L1_LOSC_4_V2-1126259446-32.gwf
N = 131072 fs = 4096.0


## 2 — Prétraitement commun identique à real.2.4

In [3]:

NOISE=((t_rel>=-12)&(t_rel<=-2))|((t_rel>=2)&(t_rel<=12))

def center(x):
    return np.asarray(x,float)-np.mean(x)

def whiten(x):
    x=center(x)
    n=len(x)
    f=np.fft.rfftfreq(n,1/fs)
    X=np.fft.rfft(x)
    fp,P=signal.welch(x[NOISE],fs=fs,nperseg=4096)
    Pi=np.interp(f,fp,P,left=P[0],right=P[-1])
    floor=np.finfo(float).eps*np.max(Pi)
    y=np.fft.irfft(X/np.sqrt(Pi+floor),n=n)
    return y/np.std(y[NOISE])

def band_notch(x,lo=35,hi=350):
    sos=signal.butter(4,[lo,hi],btype="bandpass",fs=fs,output="sos")
    y=signal.sosfiltfilt(sos,center(x))
    for f0 in [60,120,180,240,300]:
        if lo<f0<hi:
            b,a=signal.iirnotch(f0,Q=30,fs=fs)
            y=signal.filtfilt(b,a,y)
    return y

H=band_notch(whiten(h1))
L=band_notch(whiten(l1))

EVENT_WIN=(-.14,.06)
em=(t_rel>=EVENT_WIN[0])&(t_rel<=EVENT_WIN[1])
te=t_rel[em]
He=H[em]
Le=L[em]

print("event duration [s] =",te[-1]-te[0])
print("event samples =",len(te))


event duration [s] = 0.19970703125
event samples = 819


## 3 — Split train/test indépendant

On découpe la fenêtre événementielle en deux portions temporelles disjointes :

\[
\text{A}=[-0.14,-0.04]\ {\rm s},
\]

\[
\text{B}=[-0.04,+0.06]\ {\rm s}.
\]

Puis on réalise :

\[
A\rightarrow B,
\qquad
B\rightarrow A.
\]

Un second split légèrement décalé est ajouté comme contrôle de robustesse.


In [4]:

SPLITS=[
    ("split_1",(-.14,-.04),(-.04,.06)),
    ("split_2",(-.13,-.03),(-.03,.05)),
]

def extract_segment(win):
    m=(t_rel>=win[0])&(t_rel<win[1])
    return H[m],L[m]

for name,wa,wb in SPLITS:
    Ha,La=extract_segment(wa)
    Hb,Lb=extract_segment(wb)
    print(name, "A samples",len(Ha),"B samples",len(Hb))
    assert len(Ha)>200 and len(Hb)>200


split_1 A samples 410 B samples 409
split_2 A samples 410 B samples 327


## 4 — Estimateurs communs

In [5]:

MAX_LAG=.012

def parab(a,b,c):
    d=a-2*b+c
    return 0.0 if d==0 or not np.isfinite(d) else .5*(a-c)/d

def xcorr_lag(a,b):
    a=center(a); b=center(b)
    c=signal.correlate(a,b,mode="full",method="fft")
    lag=signal.correlation_lags(len(a),len(b),mode="full")/fs
    m=np.abs(lag)<=MAX_LAG
    c=c[m]; lag=lag[m]
    cn=c/(np.linalg.norm(a)*np.linalg.norm(b))
    s=np.abs(cn); i=int(np.argmax(s))
    frac=parab(s[i-1],s[i],s[i+1]) if 0<i<len(s)-1 else 0
    return float(lag[i]+frac/fs),float(cn[i])

def phat_lag(a,b):
    a=center(a); b=center(b)
    n=int(2**np.ceil(np.log2(len(a)+len(b))))
    A=np.fft.rfft(a,n=n); B=np.fft.rfft(b,n=n)
    G=A*np.conj(B)
    R=G/(np.abs(G)+np.finfo(float).eps)
    cc=np.fft.irfft(R,n=n)
    cc=np.concatenate((cc[-n//2:],cc[:n//2]))
    lag=np.arange(-n//2,n//2)/fs
    m=np.abs(lag)<=MAX_LAG
    cc=cc[m]; lag=lag[m]
    s=np.abs(cc); i=int(np.argmax(s))
    frac=parab(s[i-1],s[i],s[i+1]) if 0<i<len(s)-1 else 0
    return float(lag[i]+frac/fs),float(cc[i])

def phase_lag(a,b,fmin=40,fmax=300,cohmin=.10):
    nper=min(512,len(a))
    nover=nper//2
    f,P=signal.csd(b,a,fs=fs,nperseg=nper,noverlap=nover,window="hann")
    _,coh=signal.coherence(a,b,fs=fs,nperseg=nper,noverlap=nover,window="hann")
    ph=np.unwrap(np.angle(P))
    m=(f>=fmin)&(f<=fmax)&(coh>=cohmin)&np.isfinite(ph)
    if m.sum()<5:
        return np.nan,np.nan
    ff=f[m]; pp=ph[m]; ww=coh[m]
    A=np.c_[ff,np.ones_like(ff)]
    W=np.sqrt(ww)
    beta,*_=np.linalg.lstsq(A*W[:,None],pp*W,rcond=None)
    pred=A@beta
    mean=np.average(pp,weights=ww)
    ssr=np.sum(ww*(pp-pred)**2)
    sst=np.sum(ww*(pp-mean)**2)
    return float(beta[0]/(2*np.pi)),float(1-ssr/sst if sst>0 else np.nan)

def estimator_table(label,a,b):
    tx,cx=xcorr_lag(a,b)
    tp,cp=phat_lag(a,b)
    tf,r2=phase_lag(a,b)
    return pd.DataFrame([
        [label,"xcorr",tx*1e3,abs(cx)],
        [label,"GCC-PHAT",tp*1e3,abs(cp)],
        [label,"phase-slope",tf*1e3,r2],
    ],columns=["case","estimator","lag_ms","quality"])


## 5 — Estimation du transfert sur train uniquement

In [6]:

def transfer_estimate(a,b,nper=256,nover=192):
    f,t,Za=signal.stft(a,fs=fs,nperseg=nper,noverlap=nover,
                       window="hann",boundary=None)
    _,_,Zb=signal.stft(b,fs=fs,nperseg=nper,noverlap=nover,
                       window="hann",boundary=None)
    eps=np.finfo(float).eps
    num=np.sum(Zb*np.conj(Za),axis=1)
    den=np.sum(np.abs(Za)**2,axis=1)+eps
    T=num/den
    power=np.sum(np.abs(Za)*np.abs(Zb),axis=1)

    amp=np.abs(T)
    ph=np.unwrap(np.angle(T))
    win=15 if len(f)>=15 else max(5,(len(f)//2)*2+1)
    if win%2==0: win+=1
    poly=3 if win>=5 else 2

    amp_s=signal.savgol_filter(amp,win,poly,mode="interp")
    ph_s=signal.savgol_filter(ph,win,poly,mode="interp")
    Ts=amp_s*np.exp(1j*ph_s)

    return f,Ts,power

def apply_transfer(a,fgrid,Tgrid):
    A=np.fft.rfft(a)
    f=np.fft.rfftfreq(len(a),1/fs)
    amp=np.interp(f,fgrid,np.abs(Tgrid),left=np.abs(Tgrid[0]),right=np.abs(Tgrid[-1]))
    uph=np.unwrap(np.angle(Tgrid))
    ph=np.interp(f,fgrid,uph,left=uph[0],right=uph[-1])
    Ti=amp*np.exp(1j*ph)
    return np.fft.irfft(A*Ti,n=len(a))


## 6 — Cross-validation A→B et B→A

In [7]:

rows=[]
transfer_meta=[]

for split_name,winA,winB in SPLITS:
    HA,LA=extract_segment(winA)
    HB,LB=extract_segment(winB)

    # Baselines on held-out segments
    rows.append(estimator_table(f"{split_name}_A_baseline",HA,LA))
    rows.append(estimator_table(f"{split_name}_B_baseline",HB,LB))

    # Train A -> test B
    fA,TA,pA=transfer_estimate(HA,LA)
    HB_corr=apply_transfer(HB,fA,TA)
    rows.append(estimator_table(f"{split_name}_AtoB_corrected",HB_corr,LB))

    # Train B -> test A
    fB,TB,pB=transfer_estimate(HB,LB)
    HA_corr=apply_transfer(HA,fB,TB)
    rows.append(estimator_table(f"{split_name}_BtoA_corrected",HA_corr,LA))

    transfer_meta.append({
        "split":split_name,
        "A_window":winA,
        "B_window":winB,
        "A_transfer_amp_median":float(np.median(np.abs(TA))),
        "B_transfer_amp_median":float(np.median(np.abs(TB))),
    })

results=pd.concat(rows,ignore_index=True)
display(results)
display(pd.DataFrame(transfer_meta))


,case,estimator,lag_ms,quality
0,split_1_A_baseline,xcorr,9.006928,0.240245
1,split_1_A_baseline,GCC-PHAT,0.225318,0.183700
2,split_1_A_baseline,phase-slope,4.020131,0.327675
3,split_1_B_baseline,xcorr,-5.197061,0.270373
4,split_1_B_baseline,GCC-PHAT,-0.151370,0.518028
5,split_1_B_baseline,phase-slope,-1.466814,0.152586
6,split_1_AtoB_corrected,xcorr,-5.835318,0.245757
7,split_1_AtoB_corrected,GCC-PHAT,0.064484,0.451319
8,split_1_AtoB_corrected,phase-slope,7.802840,0.763827
9,split_1_BtoA_corrected,xcorr,11.595287,0.207256


,split,A_window,B_window,A_transfer_amp_median,B_transfer_amp_median
0,split_1,"(-0.14, -0.04)","(-0.04, 0.06)",7.047701,10.399908
1,split_2,"(-0.13, -0.03)","(-0.03, 0.05)",1.772111,10.143432


## 7 — Mesure de spread avant/après

In [8]:

def case_spread(df,case):
    z=df[df.case==case].lag_ms.to_numpy(float)
    return float(np.nanmax(z)-np.nanmin(z))

spread_rows=[]
for case in results.case.unique():
    d=results[results.case==case]
    spread_rows.append({
        "case":case,
        "spread_ms":case_spread(results,case),
        "median_quality":float(np.nanmedian(d.quality))
    })

spread_df=pd.DataFrame(spread_rows)
display(spread_df)


,case,spread_ms,median_quality
0,split_1_A_baseline,8.781611,0.240245
1,split_1_B_baseline,5.045691,0.270373
2,split_1_AtoB_corrected,13.638158,0.451319
3,split_1_BtoA_corrected,11.120358,0.199225
4,split_2_A_baseline,10.533259,0.118920
5,split_2_B_baseline,4.418047,0.352168
6,split_2_AtoB_corrected,13.749598,0.499372
7,split_2_BtoA_corrected,13.402208,0.189839


## 8 — Gains de convergence cross-validés

In [9]:

cv_rows=[]

for split_name,_,_ in SPLITS:
    A_base=float(spread_df[spread_df.case==f"{split_name}_A_baseline"].spread_ms.iloc[0])
    B_base=float(spread_df[spread_df.case==f"{split_name}_B_baseline"].spread_ms.iloc[0])
    AtoB=float(spread_df[spread_df.case==f"{split_name}_AtoB_corrected"].spread_ms.iloc[0])
    BtoA=float(spread_df[spread_df.case==f"{split_name}_BtoA_corrected"].spread_ms.iloc[0])

    red_AtoB=(B_base-AtoB)/B_base if B_base>0 else np.nan
    red_BtoA=(A_base-BtoA)/A_base if A_base>0 else np.nan

    cv_rows += [
        {
            "split":split_name,
            "direction":"A->B",
            "baseline_test_spread_ms":B_base,
            "corrected_test_spread_ms":AtoB,
            "reduction_fraction":red_AtoB
        },
        {
            "split":split_name,
            "direction":"B->A",
            "baseline_test_spread_ms":A_base,
            "corrected_test_spread_ms":BtoA,
            "reduction_fraction":red_BtoA
        }
    ]

cv_df=pd.DataFrame(cv_rows)
display(cv_df)

CV_MEDIAN_REDUCTION=float(np.nanmedian(cv_df.reduction_fraction))
CV_MIN_REDUCTION=float(np.nanmin(cv_df.reduction_fraction))
print("CV median reduction =",CV_MEDIAN_REDUCTION)
print("CV minimum reduction =",CV_MIN_REDUCTION)


,split,direction,baseline_test_spread_ms,corrected_test_spread_ms,reduction_fraction
0,split_1,A->B,5.045691,13.638158,-1.702932
1,split_1,B->A,8.781611,11.120358,-0.266323
2,split_2,A->B,4.418047,13.749598,-2.112144
3,split_2,B->A,10.533259,13.402208,-0.272371


CV median reduction = -0.987651011797168
CV minimum reduction = -2.112143851573017


## 9 — Robustesse par estimateur

In [10]:

# Compare distribution of corrected lags across all CV directions
corr_cases=[c for c in results.case.unique() if "corrected" in c]
corr=results[results.case.isin(corr_cases)].copy()

robust_rows=[]
for est in corr.estimator.unique():
    vals=corr[corr.estimator==est].lag_ms.to_numpy(float)
    robust_rows.append({
        "estimator":est,
        "median_lag_ms":float(np.nanmedian(vals)),
        "mad_lag_ms":float(np.nanmedian(np.abs(vals-np.nanmedian(vals)))),
        "min_lag_ms":float(np.nanmin(vals)),
        "max_lag_ms":float(np.nanmax(vals))
    })

robust_df=pd.DataFrame(robust_rows)
display(robust_df)


,estimator,median_lag_ms,mad_lag_ms,min_lag_ms,max_lag_ms
0,xcorr,2.717458,8.447023,-5.835318,11.595287
1,GCC-PHAT,0.047464,0.134421,-0.204358,0.474928
2,phase-slope,5.432313,2.532000,-2.343479,8.125786


## 10 — Critères de validation cross-validée

Le transfert est considéré **cross-validé** si :

1. toutes les sorties sont finies ;
2. réduction médiane du spread ≥ 30 % ;
3. réduction minimale sur les 4 directions ≥ 10 % ;
4. aucune direction n'augmente le spread de plus de 10 %.

Ces seuils sont des critères de benchmark, pas des constantes GVH.

Même si le transfert est cross-validé, ce notebook ne certifie pas encore le timing physique.


In [11]:

FINITE_OK=bool(np.isfinite(results[["lag_ms","quality"]].to_numpy()).all())

MEDIAN_REDUCTION_PASS=bool(
    np.isfinite(CV_MEDIAN_REDUCTION) and CV_MEDIAN_REDUCTION>=.30
)

MIN_REDUCTION_PASS=bool(
    np.isfinite(CV_MIN_REDUCTION) and CV_MIN_REDUCTION>=.10
)

NO_MAJOR_DEGRADATION=bool(
    np.all(cv_df.reduction_fraction.to_numpy()>=-.10)
)

TRANSFER_CROSS_VALIDATED=bool(
    FINITE_OK and
    MEDIAN_REDUCTION_PASS and
    MIN_REDUCTION_PASS and
    NO_MAJOR_DEGRADATION
)

gates={
    "FINITE_OK":FINITE_OK,
    "MEDIAN_REDUCTION_PASS":MEDIAN_REDUCTION_PASS,
    "MIN_REDUCTION_PASS":MIN_REDUCTION_PASS,
    "NO_MAJOR_DEGRADATION":NO_MAJOR_DEGRADATION,
    "TRANSFER_CROSS_VALIDATED":TRANSFER_CROSS_VALIDATED
}

print(json.dumps(gates,indent=2))


{
  "FINITE_OK": true,
  "MEDIAN_REDUCTION_PASS": false,
  "MIN_REDUCTION_PASS": false,
  "NO_MAJOR_DEGRADATION": false,
  "TRANSFER_CROSS_VALIDATED": false
}


## 11 — Verdict

In [12]:

if not FINITE_OK:
    FINAL_STATUS="UNRESOLVED"
elif TRANSFER_CROSS_VALIDATED:
    FINAL_STATUS="DETECTOR-TRANSFER-CROSS-VALIDATED"
else:
    FINAL_STATUS="DETECTOR-TRANSFER-NOT-CROSS-VALIDATED"

artifact={
    "step":"RACC-C.4-real.2.5",
    "event":"GW150914",
    "previous_notebook_md5":"8a746d753fd7084bce0c2377f01e4d21",
    "method":"bidirectional temporal cross-validation of relative H1->L1 transfer",
    "target_lag_used":False,
    "published_lag_used_for_selection":False,
    "cv_results":cv_df.to_dict(orient="records"),
    "robustness":robust_df.to_dict(orient="records"),
    "gates":gates,
    "final_status":FINAL_STATUS,
    "TIMING_VALIDATED":False,
    "D_T_ref":"NOT-AUTHORIZED",
    "REAL3_AUTHORIZED":False,
    "DISPERSION_READY_modified":False,
    "HH_chain_modified":False,
    "core_GVH_modified":False
}

print(json.dumps(artifact,indent=2))


{
  "step": "RACC-C.4-real.2.5",
  "event": "GW150914",
  "previous_notebook_md5": "8a746d753fd7084bce0c2377f01e4d21",
  "method": "bidirectional temporal cross-validation of relative H1->L1 transfer",
  "target_lag_used": false,
  "published_lag_used_for_selection": false,
  "cv_results": [
    {
      "split": "split_1",
      "direction": "A->B",
      "baseline_test_spread_ms": 5.0456912148112325,
      "corrected_test_spread_ms": 13.638157739795666,
      "reduction_fraction": -1.70293150317283
    },
    {
      "split": "split_1",
      "direction": "B->A",
      "baseline_test_spread_ms": 8.781610625427387,
      "corrected_test_spread_ms": 11.120358447145769,
      "reduction_fraction": -0.26632333423512033
    },
    {
      "split": "split_2",
      "direction": "A->B",
      "baseline_test_spread_ms": 4.418047041756759,
      "corrected_test_spread_ms": 13.749597936963657,
      "reduction_fraction": -2.112143851573017
    },
    {
      "split": "split_2",
      "direction

## 12 — Stop-rule finale

Si :

\[
\boxed{\texttt{DETECTOR-TRANSFER-CROSS-VALIDATED}}
\]

alors la réponse H1/L1 devient une explication beaucoup plus solide de la divergence inter-estimateurs.

La suite sera alors un **test final de convergence physique** avec le transfert cross-validé gelé.

Ce notebook ne modifie toujours pas :

\[
\boxed{
\texttt{TIMING-VALIDATED=False},
\quad
D_T^{ref}=\texttt{NOT-AUTHORIZED},
\quad
\texttt{REAL3-AUTHORIZED=False}
}
\]

Si le transfert échoue hors-échantillon, il ne doit pas être utilisé pour autoriser la suite.
